# Recovery: when the physical board disagrees with the engine's board

**Deliverable:** "Detect mismatches between the physical board and the engine's expected board,
and recover" (README, Perception and rules milestone). The camera is the source of truth.

Pipeline: `diff_boards(expected, observation)` -> `classify_mismatch(diff, ...)` ->
`choose_recovery_action(kind, attempts, cfg)` -> the game manager executes the action.

| Kind | Typical cause | Likely action |
|---|---|---|
| `VISION_UNCERTAIN` | every differing square has low classifier confidence | re-verify (re-read) |
| `MISPLACED` | the arm put a piece on a neighbouring square | arm retry (move it back) |
| `KNOCKED` | a piece vanished (fell over, slid off) | ask the human / arm retry |
| `ILLEGAL_HUMAN_MOVE` | the human's change matches no legal move | ask the human to fix it |
| `UNKNOWN` | anything else | ask the human |

**TODO checklist**
- [ ] `BoardObservation` helper methods (vision -> game data type; fields are fixed).
- [ ] Board diff (`diff_boards`, `BoardDiff` helpers).
- [ ] First-pass rule-based classifier (refine with real failure logs).
- [ ] Recovery policy `choose_recovery_action` (retry budget, when to ask the human).
- [ ] Arm-side corrections: compute the fix-up steps for MISPLACED / KNOCKED.
- [ ] Human prompts (LED, sound, screen, voice?) for ASK_HUMAN.
- [ ] Self-checks (plan at the bottom).

In [ ]:
from __future__ import annotations

import enum
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import chess
import yaml

GAME_CFG_PATH = Path("../../gambit_bringup/config/game.yaml")
with open(GAME_CFG_PATH) as _f:
    _CFG = yaml.safe_load(_f)
RECOVERY_CFG = _CFG["recovery"]
MIN_SQUARE_CONFIDENCE = _CFG["game"]["min_square_confidence"]

## What vision gives the game

`gambit_vision` reports a piece placement (the first field of a FEN, e.g.
`rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR`) and a confidence in 0..1 for each of the 64 squares
(index = python-chess square, a1 = 0). Optional `hand_present` lets the game ignore frames
where the human's hand is over the board (hesitation features use it too).

In [ ]:
@dataclass
class BoardObservation:
    placement: str                                  # FEN piece-placement field
    confidence: list = field(default_factory=lambda: [1.0] * 64)
    timestamp: float = field(default_factory=time.time)
    hand_present: bool = False

    def piece_map(self) -> dict:
        """{square: Piece} for the observed placement."""
        # Design: chess.BaseBoard accepts a placement string directly; use its piece_map().
        # TODO: implement
        raise NotImplementedError

    def low_confidence_squares(self, threshold: float = MIN_SQUARE_CONFIDENCE) -> set:
        """Squares whose classifier confidence is below `threshold`."""
        # TODO: implement
        raise NotImplementedError

    @classmethod
    def from_board(cls, board: chess.BaseBoard, confidence: Optional[list] = None) -> "BoardObservation":
        """A perfect observation of a known board (tests, fakes, simulation)."""
        # Design: board.board_fen() as the placement; confidence defaults to all 1.0.
        # TODO: implement
        raise NotImplementedError

## Board diff

For every square where expected and observed differ, record what was expected and what was seen.
`missing` = expected a piece, saw empty; `extra` = expected empty, saw a piece; `changed` = both
occupied but different pieces (wrong piece or wrong colour, often a vision error).

In [ ]:
@dataclass
class BoardDiff:
    missing: dict = field(default_factory=dict)     # square -> expected piece
    extra: dict = field(default_factory=dict)       # square -> observed piece
    changed: dict = field(default_factory=dict)     # square -> (expected, observed)

    @property
    def squares(self) -> set:
        """Every square that differs (union of the three dicts)."""
        # TODO: implement
        raise NotImplementedError

    def __bool__(self) -> bool:
        """True if anything differs, so `if diff:` reads naturally."""
        # TODO: implement
        raise NotImplementedError

    def __str__(self) -> str:
        """Compact text, e.g. '-Pe2 +Pe4' or 'N>Bg1'; '(no difference)' when empty."""
        # TODO: implement
        raise NotImplementedError


def diff_boards(expected: chess.BaseBoard | dict, observed: BoardObservation | chess.BaseBoard | dict) -> BoardDiff:
    """Square-by-square difference between the expected board and what the camera saw."""
    # Design:
    # - Normalise both sides to {square: Piece} maps (accept a board, a dict or a BoardObservation),
    #   so callers and tests can pass whatever they have.
    # - Walk all 64 squares (chess.SQUARES) and sort each difference into missing / extra / changed.
    # TODO: implement
    raise NotImplementedError

## Classification (first pass)

Simple, explainable rules. `human_turn=True` means the expected board is the position before the
human moved and no legal move explained the change (the game manager checks legal moves first).

In [ ]:
class MismatchKind(enum.Enum):
    NONE = "none"
    VISION_UNCERTAIN = "vision_uncertain"
    MISPLACED = "misplaced"
    KNOCKED = "knocked"
    ILLEGAL_HUMAN_MOVE = "illegal_human_move"
    UNKNOWN = "unknown"


def classify_mismatch(diff: BoardDiff, observation: Optional[BoardObservation] = None,
                      human_turn: bool = False,
                      min_conf: float = MIN_SQUARE_CONFIDENCE) -> MismatchKind:
    """Name the most likely cause of a mismatch (see the table at the top)."""
    # Design (order of the rules matters, most specific/harmless first):
    # - Empty diff -> NONE.
    # - All differing squares are low-confidence in the observation -> VISION_UNCERTAIN (re-read
    #   before blaming anyone).
    # - On the human's turn -> ILLEGAL_HUMAN_MOVE (the manager already tried every legal move).
    # - Exactly one missing + one extra of the same piece on adjacent squares
    #   (chess.square_distance <= 1) -> MISPLACED.
    # - Only missing pieces -> KNOCKED. Anything else -> UNKNOWN.
    # Decisions for you: refine these once you have real failure logs.
    # TODO: implement
    raise NotImplementedError

## Recovery policy

In [ ]:
class RecoveryAction(enum.Enum):
    ACCEPT = "accept"                # nothing to do
    REVERIFY = "reverify"            # read the board again (maybe after a short wait)
    ARM_RETRY = "arm_retry"          # let the arm fix its own mistake
    ASK_HUMAN = "ask_human"          # stop and ask the human to fix the board
    ABORT = "abort"                  # give up on the game


def choose_recovery_action(kind: MismatchKind, attempts: int, cfg: dict = RECOVERY_CFG) -> RecoveryAction:
    """Decide what to do about a mismatch.

    Inputs: the classified `kind`, how many recovery attempts were already made for this move, and
    `cfg` (`max_arm_retries`, `max_verify_attempts`, `ask_human_on_uncertain` from game.yaml).
    Output: a RecoveryAction.
    """
    # Design hints:
    # - NONE -> ACCEPT. VISION_UNCERTAIN -> REVERIFY until max_verify_attempts, then ASK_HUMAN.
    # - MISPLACED -> ARM_RETRY up to max_arm_retries, then ASK_HUMAN.
    # - ILLEGAL_HUMAN_MOVE -> ASK_HUMAN (never "fix" a human move silently).
    # - Log every decision (logger.ipynb) to measure the README retry targets.
    # Decisions for you: when (if ever) to ABORT; whether KNOCKED is safe to retry automatically.
    # TODO: implement
    raise NotImplementedError


def plan_correction(diff: BoardDiff, kind: MismatchKind) -> list:
    """Arm steps (physical_moves.Step) that turn the observed board back into the expected one.

    Return [] when no safe automatic fix exists.
    """
    # Design hints:
    # - MISPLACED: a single move_piece from the extra square back to the missing square.
    # - KNOCKED: the piece may be lying on its side or off the board: probably ASK_HUMAN instead.
    # TODO: implement
    raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - identical boards -> empty diff (falsy); start vs after e4 -> missing {e2}, extra {e4}.
# - misplaced: expected after e4, observed pawn on f4 instead -> MISPLACED.
# - knocked: knight g1 removed -> KNOCKED.
# - same knocked diff but g1 has confidence 0.3 in the observation -> VISION_UNCERTAIN.
# - vision says bishop on g1 where a knight is -> changed {g1}, UNKNOWN; with human_turn=True ->
#   ILLEGAL_HUMAN_MOVE.
# - choose_recovery_action: NONE -> ACCEPT; retry budgets respected for each kind.
# - plan_correction for a MISPLACED diff replays (physical_moves.apply_steps) to the expected board.
# Example shape of a test:
# b = chess.Board(); assert not diff_boards(b, BoardObservation.from_board(b))
print("Self-check: not written yet (see plan above)")